# Treinamento do detector de EPIs de solda (YOLOv8)

**Projeto:** Verificador de EPIs de Solda com Visão Computacional (FSG + Marcopolo)
**Etapa:** 1.2 – Dataset Welding-PPE + treinamento (RF31, RF32)

O que este notebook faz, em ordem:

1. Baixa o dataset **Welding-PPE** (a sua cópia/fork no Roboflow) via API.
2. Confere se a versão baixada **não** é de "objetos isolados" (recortes), o que estragaria o treino.
3. **Remapeia** as 9 classes originais para as 5 classes internas do projeto e reescreve os labels e o `data.yaml`.
4. Gera o **relatório de balanceamento** (instâncias por classe).
5. Treina **YOLOv8n** e **YOLOv8s** com `imgsz` **640** e **960** e compara os resultados.
6. Avalia no conjunto de **teste**: precisão, recall, mAP@0.5 e mAP@0.5:0.95 **por classe** + matriz de confusão.
7. Exporta o melhor modelo como `epi_solda_best.pt`.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.

### Atribuição do dataset (CC BY 4.0)
Dataset "Welding-PPE", por *Welding PPE*, Roboflow Universe, licença CC BY 4.0.
https://universe.roboflow.com/welding-ppe/welding-ppe

```bibtex
@misc{ welding-ppe_dataset,
  title = { Welding-PPE Dataset },
  type = { Open Source Dataset },
  author = { Welding PPE },
  howpublished = { \url{ https://universe.roboflow.com/welding-ppe/welding-ppe } },
  url = { https://universe.roboflow.com/welding-ppe/welding-ppe },
  journal = { Roboflow Universe },
  publisher = { Roboflow },
  year = { 2023 },
  month = { jun },
  note = { visited on 2026-10-10 },
}
```

## 0. Por que usamos um *fork* e não a versão pública do dataset

As 3 versões públicas do Welding-PPE (v1, v2, v3) foram geradas com o pré-processamento
**"Isolate Objects"**. Ele transforma **cada caixa anotada em uma imagem separada** (um recorte só do objeto).

Para o nosso caso isso é ruim:
- o modelo aprenderia a reconhecer uma luva **ocupando a imagem inteira**, mas na câmera do posto a luva é um objeto **pequeno dentro da cena**;
- o conjunto de teste também seria de recortes, e as métricas sairiam **otimistas e enganosas**;
- todas as versões foram redimensionadas para 640×640 ("Stretch"), então treinar com `imgsz=960` não ganharia detalhe nenhum.

**Solução:** fazer um *fork* (cópia gratuita) do projeto para a sua conta e gerar uma versão nova, com as
**801 imagens originais**, sem "Isolate Objects" e sem redimensionar. O passo a passo está em `docs/dataset_welding_ppe.md`.

## 1. Configuração

Preencha `WORKSPACE` e `VERSAO` com os dados do **seu fork** (aparecem na URL do projeto no Roboflow:
`app.roboflow.com/<workspace>/<projeto>/<versao>`).

A chave da API **não** vai no código: cadastre-a no ícone 🔑 *Secrets* (barra lateral do Colab) com o nome
`ROBOFLOW_API_KEY` e ative o acesso para este notebook.

In [ ]:
# ===== Dados do SEU fork no Roboflow =====
WORKSPACE = "SEU-WORKSPACE"     # ex.: "fulano-xyz"
PROJETO   = "welding-ppe"       # nome do projeto no fork (normalmente igual)
VERSAO    = 1                   # versão que você gerou no fork

# ===== Treino =====
EXPERIMENTOS = [                # (modelo base, imgsz) – RF32 pede n e s, 640 e 960
    ("yolov8n.pt", 640),
    ("yolov8n.pt", 960),
    ("yolov8s.pt", 640),
    ("yolov8s.pt", 960),
]
EPOCAS    = 100                 # máximo de épocas
PACIENCIA = 25                  # para antes se o mAP de validação não melhorar por N épocas
SEMENTE   = 42                  # resultados reproduzíveis

# Versões fixadas (as mesmas do requirements.txt do projeto)
VERSAO_ULTRALYTICS = "8.4.175"
VERSAO_ROBOFLOW    = "1.7.1"

# Pasta no Google Drive onde os treinos ficam salvos (sobrevive à desconexão do Colab)
PASTA_DRIVE = "/content/drive/MyDrive/epi_solda"

## 2. Ambiente: GPU, bibliotecas e Google Drive

Os treinos ficam no Google Drive porque a sessão gratuita do Colab pode desconectar.
Se cair no meio, basta rodar o notebook de novo: os experimentos já concluídos são pulados e o
que estava pela metade é **retomado** do último checkpoint.

In [ ]:
import subprocess, sys

# Instala as versões fixadas (silencioso)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"ultralytics=={VERSAO_ULTRALYTICS}", f"roboflow=={VERSAO_ROBOFLOW}"], check=True)

import torch
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENÇÃO: sem GPU! Ative em Ambiente de execução → Alterar tipo → GPU. Em CPU o treino leva horas.")

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PASTA_DRIVE = Path(PASTA_DRIVE)
PASTA_TREINOS = PASTA_DRIVE / "treinos"
PASTA_TREINOS.mkdir(parents=True, exist_ok=True)
print("Treinos serão salvos em:", PASTA_TREINOS)

## 3. Download do dataset via API do Roboflow (RF31)

Baixamos no formato **YOLOv8**: uma pasta por split (`train/`, `valid/`, `test/`), cada uma com
`images/` e `labels/`. Cada arquivo `.txt` de label tem uma linha por objeto:

```
<id_da_classe> <x_centro> <y_centro> <largura> <altura>     (valores de 0 a 1, relativos à imagem)
```

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

chave = userdata.get("ROBOFLOW_API_KEY")       # lida do cofre de Secrets, nunca escrita no código
rf = Roboflow(api_key=chave)
versao = rf.workspace(WORKSPACE).project(PROJETO).version(VERSAO)
dataset = versao.download("yolov8", location="/content/welding_ppe_original", overwrite=True)

PASTA_ORIGINAL = Path(dataset.location)
print("Dataset baixado em:", PASTA_ORIGINAL)
print((PASTA_ORIGINAL / "data.yaml").read_text(encoding="utf-8"))

## 4. Funções auxiliares de leitura do dataset

In [ ]:
import yaml

SPLITS = {"train": "train", "val": "valid", "test": "test"}   # nome Ultralytics -> pasta Roboflow


def ler_nomes_classes(pasta: Path) -> list[str]:
    # Lê a lista de nomes de classes do data.yaml (pode ser lista ou dicionário {id: nome}).
    dados = yaml.safe_load((pasta / "data.yaml").read_text(encoding="utf-8"))
    nomes = dados["names"]
    if isinstance(nomes, dict):
        nomes = [nomes[i] for i in sorted(nomes)]
    return list(nomes)


def splits_existentes(pasta: Path) -> dict[str, Path]:
    # Retorna {split: pasta} só para os splits que existem no download.
    return {s: pasta / p for s, p in SPLITS.items() if (pasta / p / "labels").is_dir()}


def ler_label(arquivo: Path) -> list[list[float]]:
    # Lê um .txt YOLO e devolve uma lista de linhas numéricas [classe, ...coordenadas].
    linhas = []
    for linha in arquivo.read_text(encoding="utf-8").splitlines():
        partes = linha.split()
        if partes:
            linhas.append([int(partes[0])] + [float(v) for v in partes[1:]])
    return linhas


def caixa_de(linha: list[float]) -> tuple[float, float, float, float]:
    # Devolve (xc, yc, w, h). Se a linha for polígono (segmentação), converte para a caixa envolvente.
    valores = linha[1:]
    if len(valores) == 4:
        return tuple(valores)
    xs, ys = valores[0::2], valores[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)


print("Classes originais:", ler_nomes_classes(PASTA_ORIGINAL))
print("Splits encontrados:", list(splits_existentes(PASTA_ORIGINAL)))

## 5. Verificação: a versão é de "objetos isolados"?

Se a maioria das imagens tiver **uma única caixa ocupando quase a imagem toda**, a versão pode ter sido gerada com
"Isolate Objects" e **não serviria** para o nosso detector (ver seção 0).
É só um **aviso**: o Welding-PPE tem muitas fotos de produto (uma luva ou bota sobre fundo neutro), que também
caem nessa regra. A confirmação definitiva é no Roboflow: em *Preprocessing* deve aparecer só "Auto-Orient".

In [ ]:
def verificar_objetos_isolados(pasta: Path) -> float:
    # Retorna a fração de imagens de treino com 1 só caixa cobrindo mais de 60% da imagem.
    labels = list((pasta / "train" / "labels").glob("*.txt"))
    suspeitas = 0
    for arq in labels:
        linhas = ler_label(arq)
        if len(linhas) == 1:
            _, _, w, h = caixa_de(linhas[0])
            if w * h > 0.60:
                suspeitas += 1
    return suspeitas / max(len(labels), 1)


fracao = verificar_objetos_isolados(PASTA_ORIGINAL)
print(f"Imagens com 1 caixa ocupando >60% da imagem: {fracao:.0%}")
if fracao > 0.5:
    # Aviso (não erro): o Welding-PPE tem muitas fotos de produto com um único objeto grande,
    # então a fração pode ser alta mesmo numa versão correta. Confirme no Roboflow que a
    # versão mostra apenas "Auto-Orient" em Preprocessing.
    print("ATENÇÃO: muitas imagens com um único objeto grande. Confira no Roboflow se a versão NÃO "
          "tem 'Isolate Objects' em Preprocessing (ver docs/dataset_welding_ppe.md).")
else:
    print("OK: maioria das imagens com cena completa.")

## 6. Remapeamento das classes (RF03, RF31)

O dataset tem **9 classes, algumas duplicadas** com nomes diferentes. Reescrevemos cada label para as
**5 classes internas** do projeto. Assim o modelo treinado já "fala" a língua do sistema.

| Classe original | Classe interna | Observação |
|---|---|---|
| Welding Mask | `mascara` | |
| helmet | `mascara` | **verificado visualmente**: são capacetes/máscaras de solda metálicos com visor escuro, não capacetes de segurança |
| Welding Gloves, welding-glove | `luvas` | duplicadas |
| Welding Apron, apron | `avental` | duplicadas |
| Welding Coverall | `macacao` | separado do avental para permitir `aceitar_macacao=false` (plano, item 5.3) |
| Safety Shoes, welding-shose | `calcado` | duplicadas ("shose" é erro de digitação do autor) |

A classe interna `capacete` (capacete de segurança) **não existe** neste dataset. O EPI opcional "capacete"
só poderá ser ativado depois de um fine-tuning com fotos anotadas (RF33).

Se você já remapeou as classes no próprio Roboflow, os nomes internos também são aceitos.

In [ ]:
import shutil

CLASSES_INTERNAS = ["mascara", "luvas", "avental", "macacao", "calcado"]

# Nome original (minúsculo, sem espaços nas pontas) -> classe interna
MAPEAMENTO = {
    "welding mask": "mascara",
    "helmet": "mascara",
    "welding gloves": "luvas",
    "welding-glove": "luvas",
    "welding apron": "avental",
    "apron": "avental",
    "welding coverall": "macacao",
    "safety shoes": "calcado",
    "welding-shose": "calcado",
}
MAPEAMENTO.update({c: c for c in CLASSES_INTERNAS})   # aceita nomes já remapeados no Roboflow


def remapear_dataset(origem: Path, destino: Path) -> dict[str, int]:
    # Copia as imagens e reescreve os labels com os ids das classes internas.
    # Retorna um contador de classes originais descartadas (não mapeadas).
    nomes_originais = ler_nomes_classes(origem)
    id_novo = {}
    for i, nome in enumerate(nomes_originais):
        interno = MAPEAMENTO.get(nome.strip().lower())
        id_novo[i] = CLASSES_INTERNAS.index(interno) if interno else None

    descartadas: dict[str, int] = {}
    if destino.exists():
        shutil.rmtree(destino)
    for split, pasta_split in splits_existentes(origem).items():
        (destino / SPLITS[split] / "labels").mkdir(parents=True)
        shutil.copytree(pasta_split / "images", destino / SPLITS[split] / "images")
        for arq in (pasta_split / "labels").glob("*.txt"):
            novas = []
            for linha in ler_label(arq):
                novo = id_novo.get(linha[0])
                if novo is None:
                    nome = nomes_originais[linha[0]]
                    descartadas[nome] = descartadas.get(nome, 0) + 1
                    continue
                xc, yc, w, h = caixa_de(linha)
                novas.append(f"{novo} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            (destino / SPLITS[split] / "labels" / arq.name).write_text("\n".join(novas), encoding="utf-8")

    data_yaml = {"path": str(destino), "names": {i: n for i, n in enumerate(CLASSES_INTERNAS)}}
    for split in splits_existentes(destino):
        data_yaml[split] = f"{SPLITS[split]}/images"
    (destino / "data.yaml").write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False),
                                       encoding="utf-8")
    return descartadas


PASTA_DATASET = Path("/content/welding_ppe_remap")
descartadas = remapear_dataset(PASTA_ORIGINAL, PASTA_DATASET)
DATA_YAML = PASTA_DATASET / "data.yaml"

print("Classes internas:", CLASSES_INTERNAS)
print("Caixas descartadas (classe não mapeada):", descartadas or "nenhuma")
print(DATA_YAML.read_text())

## 7. Relatório de balanceamento (RF31)

Por que importa: se uma classe tem poucas instâncias, o modelo a aprende mal (recall baixo).
Também medimos o **tamanho médio das caixas**: objetos pequenos (luvas, calçados) são mais difíceis e
se beneficiam de `imgsz` maior.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def relatorio_balanceamento(pasta: Path) -> pd.DataFrame:
    # Conta instâncias e imagens por classe e por split, e o tamanho médio das caixas.
    registros = []
    for split, pasta_split in splits_existentes(pasta).items():
        arquivos = list((pasta_split / "labels").glob("*.txt"))
        for arq in arquivos:
            linhas = ler_label(arq)
            if not linhas:
                registros.append({"split": split, "classe": "(sem objetos)", "imagem": arq.stem, "area": None})
            for linha in linhas:
                _, _, w, h = caixa_de(linha)
                registros.append({"split": split, "classe": CLASSES_INTERNAS[linha[0]],
                                  "imagem": arq.stem, "area": w * h})
    df = pd.DataFrame(registros)
    instancias = df.pivot_table(index="classe", columns="split", values="imagem", aggfunc="count", fill_value=0)
    imagens = df.groupby("classe")["imagem"].nunique().rename("imagens_com_a_classe")
    area = (df.groupby("classe")["area"].mean() * 100).round(2).rename("area_media_%_da_imagem")
    tabela = instancias.join(imagens).join(area)
    tabela["total_instancias"] = instancias.sum(axis=1)
    return tabela.sort_values("total_instancias", ascending=False)


balanceamento = relatorio_balanceamento(PASTA_DATASET)
display(balanceamento)
balanceamento.to_csv(PASTA_DRIVE / "balanceamento_classes.csv")

ax = balanceamento.drop(index="(sem objetos)", errors="ignore")["total_instancias"].plot.bar(
    title="Instâncias por classe (todos os splits)", rot=0)
ax.set_ylabel("instâncias")
plt.tight_layout(); plt.savefig(PASTA_DRIVE / "balanceamento_classes.png"); plt.show()

### Amostras após o remapeamento
Confira visualmente se as caixas e os nomes fazem sentido (em especial `mascara`, que agora inclui o antigo `helmet`).

In [ ]:
import random
import cv2

CORES = [(220, 53, 69), (13, 110, 253), (25, 135, 84), (255, 193, 7), (111, 66, 193)]


def mostrar_amostras(pasta: Path, por_classe: int = 3) -> None:
    # Mostra algumas imagens de treino de cada classe interna, com as caixas desenhadas.
    random.seed(SEMENTE)
    labels = list((pasta / "train" / "labels").glob("*.txt"))
    fig, eixos = plt.subplots(len(CLASSES_INTERNAS), por_classe, figsize=(4 * por_classe, 3.2 * len(CLASSES_INTERNAS)))
    for c, classe in enumerate(CLASSES_INTERNAS):
        com_classe = [a for a in labels if any(l[0] == c for l in ler_label(a))]
        for j in range(por_classe):
            ax = eixos[c][j]; ax.axis("off")
            if j >= len(com_classe):
                continue
            arq = random.choice(com_classe)
            img_path = next((pasta / "train" / "images").glob(arq.stem + ".*"))
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            alt, larg = img.shape[:2]
            for linha in ler_label(arq):
                xc, yc, w, h = caixa_de(linha)
                p1 = (int((xc - w / 2) * larg), int((yc - h / 2) * alt))
                p2 = (int((xc + w / 2) * larg), int((yc + h / 2) * alt))
                cv2.rectangle(img, p1, p2, CORES[linha[0]], 3)
                cv2.putText(img, CLASSES_INTERNAS[linha[0]], (p1[0], max(p1[1] - 5, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, CORES[linha[0]], 2)
            ax.imshow(img); ax.set_title(f"{classe}: {arq.stem[:25]}", fontsize=8)
    plt.tight_layout(); plt.show()


mostrar_amostras(PASTA_DATASET)

## 8. Treinamento (RF32)

### Augmentations (o que cada uma faz e por que ajuda na solda)

*Augmentation* é gerar, durante o treino, variações artificiais das imagens. O modelo vê "imagens novas"
a cada época e generaliza melhor, o que é essencial com um dataset pequeno (~800 imagens).

| Augmentation | Onde | Parâmetro | Por que ajuda no posto de solda |
|---|---|---|---|
| Brilho/exposição | Roboflow (fork) + Ultralytics | Exposure ±25% no fork; `hsv_v=0.5` | O arco e a iluminação da fábrica mudam muito o brilho |
| Contraste/saturação | Ultralytics | `hsv_s=0.6` | Fumos "lavam" as cores; os aventais de raspa variam de tom |
| Desfoque (blur) | Roboflow (fork) | Blur até 2,5 px | Fumos, respingos no vidro da câmera, movimento |
| Mosaic | Ultralytics | `mosaic=1.0` | Junta 4 imagens em uma: mais objetos e contextos por imagem, e objetos menores |
| Scale | Ultralytics | `scale=0.5` | Simula o soldador mais perto ou mais longe da câmera (2 a 6 m) |
| Translate / rotação leve | Ultralytics | `translate=0.1`, `degrees=5` | Câmeras instaladas com ângulos levemente diferentes |
| Flip horizontal | Ultralytics | `fliplr=0.5` | Soldador destro ou canhoto, câmera à esquerda ou à direita |
| Flip vertical | – | `flipud=0` (desligado) | Pessoas de cabeça para baixo não existem no posto |
| Copy-paste | – | `copy_paste=0` | **Não tem efeito aqui:** no Ultralytics ele só funciona com máscaras de segmentação, e este dataset tem só caixas |

O Roboflow aplica as augmentations **apenas no split de treino**. Validação e teste ficam com as imagens originais, então as métricas não são infladas.

`close_mosaic=10` desliga o mosaic nas 10 últimas épocas para o modelo se ajustar a imagens "normais".

### Modelos e tamanhos
- **YOLOv8n** (*nano*, ~3 M parâmetros): rápido em CPU, é o candidato para rodar no notebook do projeto.
- **YOLOv8s** (*small*, ~11 M parâmetros): mais preciso, cerca de 2 a 3 vezes mais lento.
- **imgsz 640 x 960**: 960 enxerga melhor luvas e calçados (objetos pequenos), mas custa ~2,2× mais processamento.

In [ ]:
from ultralytics import YOLO

PARAMETROS_AUG = dict(
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    degrees=5.0, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, close_mosaic=10, mixup=0.0, copy_paste=0.0,
)


def nome_experimento(modelo: str, imgsz: int) -> str:
    return f"{Path(modelo).stem}_{imgsz}"


def treinar(modelo: str, imgsz: int) -> Path:
    # Treina um experimento (ou retoma, se foi interrompido). Retorna a pasta do experimento.
    nome = nome_experimento(modelo, imgsz)
    pasta = PASTA_TREINOS / nome
    ultimo = pasta / "weights" / "last.pt"
    marcador = pasta / "CONCLUIDO.txt"     # gravado só quando o treino termina de verdade

    if marcador.exists():
        print(f"[{nome}] já concluído, pulando.")
        return pasta
    if ultimo.exists():
        print(f"[{nome}] retomando do último checkpoint...")
        try:
            YOLO(str(ultimo)).train(resume=True)
        except Exception as erro:  # o Ultralytics recusa retomar um treino que já terminou
            print(f"[{nome}] nada a retomar ({erro}).")
        marcador.write_text("ok", encoding="utf-8")
        return pasta

    print(f"[{nome}] treinando...")
    YOLO(modelo).train(
        data=str(DATA_YAML), imgsz=imgsz, epochs=EPOCAS, patience=PACIENCIA,
        batch=-1,                 # tamanho de lote automático (~60% da memória da GPU)
        seed=SEMENTE, deterministic=True,
        project=str(PASTA_TREINOS), name=nome, exist_ok=True,
        workers=2, plots=True, **PARAMETROS_AUG,
    )
    marcador.write_text("ok", encoding="utf-8")
    return pasta


for modelo, imgsz in EXPERIMENTOS:
    treinar(modelo, imgsz)

## 9. Avaliação no conjunto de TESTE (RF32)

O conjunto de **teste** nunca foi usado no treino nem na escolha da melhor época, por isso é a medida honesta.

- **Precisão (P):** das caixas que o modelo previu, quantas estavam certas.
- **Recall (R):** dos EPIs que existiam, quantos o modelo encontrou. É o mais importante para não deixar passar um EPI presente (e evitar acusar ausência por engano).
- **mAP@0.5:** qualidade média considerando acerto quando a caixa sobrepõe ≥ 50% a caixa verdadeira. É a métrica das METAS DE VALIDAÇÃO.
- **mAP@0.5:0.95:** a mesma ideia, mas exigindo sobreposições cada vez mais precisas (mais rigorosa).

A **proteção do corpo** do sistema é avental **ou** macacão. Por isso mostramos as duas classes separadas
e também uma estimativa combinada (média ponderada pelo número de instâncias) para comparar com a meta.

In [ ]:
METAS = {  # mAP@0.5 mínimo e meta boa (prompt do projeto)
    "mascara": (0.60, 0.75), "protecao_corpo": (0.60, 0.75),
    "luvas": (0.50, 0.65), "calcado": (0.40, 0.55),
}
split_avaliacao = "test" if "test" in splits_existentes(PASTA_DATASET) else "val"
if split_avaliacao == "val":
    print("ATENÇÃO: o dataset não tem split de teste; avaliando na validação (métricas otimistas).")


# Instâncias por classe no split avaliado (contadas direto dos labels)
instancias_split = {c: 0 for c in CLASSES_INTERNAS}
for arq in (PASTA_DATASET / SPLITS[split_avaliacao] / "labels").glob("*.txt"):
    for linha in ler_label(arq):
        instancias_split[CLASSES_INTERNAS[linha[0]]] += 1


def avaliar(pasta_exp: Path, imgsz: int) -> pd.DataFrame:
    # Avalia o best.pt no split de teste e devolve uma tabela por classe.
    modelo = YOLO(str(pasta_exp / "weights" / "best.pt"))
    m = modelo.val(data=str(DATA_YAML), split=split_avaliacao, imgsz=imgsz, plots=True,
                   project=str(pasta_exp), name=f"avaliacao_{split_avaliacao}", exist_ok=True)
    linhas = []
    for i, id_classe in enumerate(m.box.ap_class_index):
        p, r, map50, map5095 = m.box.class_result(i)
        classe = m.names[int(id_classe)]
        linhas.append({"classe": classe, "instancias": instancias_split.get(classe, 0),
                       "precisao": p, "recall": r, "mAP50": map50, "mAP50_95": map5095})
    df = pd.DataFrame(linhas).set_index("classe")
    corpo = df.loc[df.index.intersection(["avental", "macacao"])]
    if len(corpo):
        pesos = corpo["instancias"] / corpo["instancias"].sum()
        df.loc["protecao_corpo*"] = [corpo["instancias"].sum()] + [(corpo[c] * pesos).sum()
                                     for c in ["precisao", "recall", "mAP50", "mAP50_95"]]
    df.loc["TODAS (média)"] = [df["instancias"].sum(), m.box.mp, m.box.mr, m.box.map50, m.box.map]
    df.attrs["pasta_avaliacao"] = Path(m.save_dir)
    return df.round(3)


def situacao_meta(classe: str, map50: float) -> str:
    chave = classe.rstrip("*")
    if chave not in METAS:
        return ""
    minimo, boa = METAS[chave]
    return "META BOA" if map50 >= boa else ("OK (mínimo)" if map50 >= minimo else "ABAIXO DO MÍNIMO")


resultados = {}
for modelo, imgsz in EXPERIMENTOS:
    nome = nome_experimento(modelo, imgsz)
    tabela = avaliar(PASTA_TREINOS / nome, imgsz)
    tabela["situacao"] = [situacao_meta(c, v) for c, v in zip(tabela.index, tabela["mAP50"])]
    resultados[nome] = tabela
    print(f"\n===== {nome} ({split_avaliacao}) =====")
    display(tabela)
    tabela.to_csv(PASTA_TREINOS / nome / f"metricas_por_classe_{split_avaliacao}.csv")

### Comparação entre os experimentos (mAP@0.5 por classe)

In [ ]:
comparacao = pd.DataFrame({nome: t["mAP50"] for nome, t in resultados.items()})
display(comparacao)
comparacao.to_csv(PASTA_DRIVE / "comparacao_experimentos_mAP50.csv")
comparacao.drop(index="TODAS (média)", errors="ignore").plot.bar(rot=0, figsize=(10, 4),
                                                                 title=f"mAP@0.5 por classe ({split_avaliacao})")
plt.axhline(0.5, ls="--", c="gray", lw=0.8); plt.ylim(0, 1); plt.tight_layout()
plt.savefig(PASTA_DRIVE / "comparacao_experimentos_mAP50.png"); plt.show()

### Matrizes de confusão

Leitura: cada coluna é a classe **verdadeira** e cada linha a classe **prevista** (normalizada).
A linha/coluna **background** mostra objetos não detectados (falsos negativos) e detecções onde não havia nada (falsos positivos).
Confusões entre `avental` e `macacao` são aceitáveis (ambos contam como proteção do corpo).

In [ ]:
from IPython.display import Image as ImagemIPy

for nome, tabela in resultados.items():
    figura = tabela.attrs["pasta_avaliacao"] / "confusion_matrix_normalized.png"
    print(nome)
    display(ImagemIPy(filename=str(figura), width=650))

## 10. Escolha e exportação do modelo

Critério sugerido: o **menor** modelo/`imgsz` que atinge os **mínimos** das metas, porque ele vai rodar em **CPU**.
O FPS real em CPU é medido depois, no notebook do projeto (`run.py --benchmark`, Etapa 1.3).

Ajuste `ESCOLHIDO` se preferir outro experimento.

In [ ]:
def pontuacao(tabela: pd.DataFrame) -> float:
    # Média do mAP@0.5 das classes que contam para as metas.
    classes = [c for c in tabela.index if c.rstrip("*") in METAS]
    return float(tabela.loc[classes, "mAP50"].mean())


ranking = sorted(resultados, key=lambda n: pontuacao(resultados[n]), reverse=True)
for n in ranking:
    print(f"{n:15s}  média mAP50 (classes das metas) = {pontuacao(resultados[n]):.3f}")

ESCOLHIDO = ranking[0]          # troque manualmente se quiser, ex.: "yolov8n_640"
print("\nModelo escolhido:", ESCOLHIDO)

destino = PASTA_DRIVE / "epi_solda_best.pt"
shutil.copy(PASTA_TREINOS / ESCOLHIDO / "weights" / "best.pt", destino)

# Pacote para levar ao projeto: modelo + métricas + gráficos principais
pacote = Path("/content/resultado_treino"); shutil.rmtree(pacote, ignore_errors=True); pacote.mkdir()
shutil.copy(destino, pacote)
for arquivo in ["balanceamento_classes.csv", "balanceamento_classes.png",
                "comparacao_experimentos_mAP50.csv", "comparacao_experimentos_mAP50.png"]:
    if (PASTA_DRIVE / arquivo).exists():
        shutil.copy(PASTA_DRIVE / arquivo, pacote)
for nome, tabela in resultados.items():
    (pacote / nome).mkdir()
    shutil.copy(PASTA_TREINOS / nome / f"metricas_por_classe_{split_avaliacao}.csv", pacote / nome)
    shutil.copy(PASTA_TREINOS / nome / "results.csv", pacote / nome)
    shutil.copy(tabela.attrs["pasta_avaliacao"] / "confusion_matrix_normalized.png", pacote / nome)
(pacote / "LEIA-ME.txt").write_text(
    f"Modelo escolhido: {ESCOLHIDO}\nClasses: {CLASSES_INTERNAS}\nSplit avaliado: {split_avaliacao}\n"
    f"Dataset: Welding-PPE (Roboflow Universe, CC BY 4.0), fork {WORKSPACE}/{PROJETO} v{VERSAO}\n",
    encoding="utf-8")
shutil.make_archive("/content/resultado_treino", "zip", pacote)

from google.colab import files
files.download("/content/resultado_treino.zip")

## 11. Próximos passos (no seu computador)

1. Descompacte `resultado_treino.zip`.
2. Copie `epi_solda_best.pt` para a pasta `models/` do projeto (`Z:\Documentos\SafeWork\models\`).
   Os arquivos `.pt` ficam fora do git (estão no `.gitignore`).
3. Coloque o restante do zip em `docs/resultados_treino/` e me avise: eu analiso as métricas por classe
   contra as METAS DE VALIDAÇÃO.
4. Baixe também o modelo de pose para uso offline: `yolov8n-pose.pt` (o `run.py` da Etapa 1.3 traz um comando para isso).